# 02 · Data Preprocessing

**Round 1 walkthrough:** show how GPS records are validated, how timestamp order and gaps are handled, and how the road graph is checked. Cleaning preserves trajectory order and never edits the supplied files.

In [1]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
DATA=ROOT/"data"/"Student folder"
print("Project:",ROOT,"| data folder available:",DATA.exists())

Project: /Users/yashas/Documents/Codex/2026-11-18/referenced-chatgpt-conversation-this-is-an-2/ML_hackathon | data folder available: True


In [2]:
import numpy as np
import pandas as pd
from pathsnap.data import read_trip
trip=read_trip(DATA/"train"/"00000001","train")
print({"input_GPS_rows":trip.raw_track_rows,"rows_after_cleaning":len(trip.track),"rows_removed":trip.raw_track_rows-len(trip.track),"nonincreasing_steps":int(np.sum(np.diff(trip.track[:,2])<=0)),"timestamp_gap_over_5s":int(np.sum(np.diff(trip.track[:,2])>5)),"coordinate_min_lon":float(trip.track[:,0].min()),"coordinate_max_lon":float(trip.track[:,0].max()),"coordinate_min_lat":float(trip.track[:,1].min()),"coordinate_max_lat":float(trip.track[:,1].max())})
display(pd.DataFrame(trip.track[:8],columns=["longitude","latitude","timestamp_s"]))

{'input_GPS_rows': 1315, 'rows_after_cleaning': 1315, 'rows_removed': 0, 'nonincreasing_steps': 0, 'timestamp_gap_over_5s': 1, 'coordinate_min_lon': 8.570524, 'coordinate_max_lon': 8.64786, 'coordinate_min_lat': 49.844053, 'coordinate_max_lat': 50.211882}


,longitude,latitude,timestamp_s
0,8.595203,49.844053,1.0
1,8.595294,49.844441,2.0
2,8.595352,49.844883,3.0
3,8.595390,49.845379,4.0
4,8.595587,49.845750,5.0
5,8.595555,49.846167,6.0
6,8.595744,49.846565,7.0
7,8.595733,49.846931,8.0


## Cleaning and transformation choices

1. Require finite coordinates and timestamps and legal longitude/latitude ranges.
2. Drop duplicate or backward timestamps while preserving the observed order; do not sort GPS points into a different path.
3. Keep long gaps and divide displacement by actual elapsed time when computing speed.
4. Convert coordinates to a local metre-scale frame before computing physical distances.
5. Exclude arcs with invalid node references or zero geometric length from candidate geometry.

The supplied archive currently has zero invalid coordinates and zero non-increasing timestamps, so the cleaning rules remove no rows in this run.

In [3]:
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
arcs=trip.arcs; n=len(trip.nodes)
valid=(arcs[:,0]>=0)&(arcs[:,1]>=0)&(arcs[:,0]<n)&(arcs[:,1]<n)
valid_arcs=arcs[valid].astype(int)
degree=np.bincount(valid_arcs.ravel(),minlength=n) if len(valid_arcs) else np.zeros(n,dtype=int)
graph=coo_matrix((np.ones(len(valid_arcs)),(valid_arcs[:,0],valid_arcs[:,1])),shape=(n,n)).tocsr()
components,labels=connected_components(graph,directed=False,return_labels=True)
component_sizes=np.bincount(labels)
print({"nodes":n,"arcs":len(arcs),"invalid_arc_rows":int((~valid).sum()),"isolated_nodes":int((degree==0).sum()),"weak_components":int(components),"largest_component_nodes":int(component_sizes.max()),"largest_component_fraction":round(float(component_sizes.max()/n),4)})

{'nodes': 88809, 'arcs': 165790, 'invalid_arc_rows': 0, 'isolated_nodes': 0, 'weak_components': 175, 'largest_component_nodes': 86854, 'largest_component_fraction': 0.978}


A weakly connected component treats a directed road arc as an undirected connection for this first connectivity check. The predictor still retains directed arc orientation in its features and transition rule.